In [3]:
# import cv2
# import numpy as np
# from pdf2image import convert_from_path
# import easyocr
# from PIL import Image
# import json

# def preprocess_image(image):
#     # Convert PIL image to OpenCV format
#     img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
#     # Denoising: Gaussian Blur and Median Filtering
#     img = cv2.GaussianBlur(img, (5, 5), 0)
#     img = cv2.medianBlur(img, 3)
    
#     # Convert to grayscale
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
#     # Binarization using Otsu's thresholding
#     _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
#     # Convert back to PIL format for OCR
#     return Image.fromarray(binary)

# def perform_ocr(image, languages=['ne', 'en']):
#     # Initialize EasyOCR reader for Nepali and English
#     reader = easyocr.Reader(languages, gpu=False)  # Set gpu=True if CUDA is available
    
#     # Perform OCR
#     results = reader.readtext(np.array(image), detail=1)
    
#     # Format results: [(bbox, text, confidence), ...]
#     output = []
#     for (bbox, text, prob) in results:
#         output.append({
#             'bounding_box': [[int(x), int(y)] for x, y in bbox],
#             'text': text,
#             'confidence': float(prob)
#         })
    
#     return output

# def process_document(input_path, output_json_path):
#     try:
#         # Check if input is PDF or image
#         if input_path.lower().endswith('.pdf'):
#             # Convert PDF to images (300 DPI)
#             images = convert_from_path(input_path, dpi=300)
#         else:
#             # Load single image
#             images = [Image.open(input_path)]
        
#         all_results = []
        
#         # Process each page/image
#         for i, image in enumerate(images):
#             # Preprocess image
#             processed_image = preprocess_image(image)
            
#             # Perform OCR
#             ocr_results = perform_ocr(processed_image)
            
#             # Add page results
#             all_results.append({
#                 'page': i + 1,
#                 'ocr_results': ocr_results
#             })
        
#         # Save results to JSON
#         with open(output_json_path, 'w', encoding='utf-8') as f:
#             json.dump(all_results, f, ensure_ascii=False, indent=2)
        
#         return all_results
    
#     except Exception as e:
#         return {'error': str(e)}

# if __name__ == "__main__":
#     # Example usage
#     input_file = "document.pdf"  # Replace with your PDF/image path
#     output_file = "ocr_output.json"
#     results = process_document(input_file, output_file)
    
#     # Print results
#     for page in results:
#         print(f"\nPage {page['page']}:")
#         for result in page['ocr_results']:
#             print(f"Text: {result['text']}, Confidence: {result['confidence']:.2f}")

In [2]:
import cv2
import numpy as np
import pytesseract
from PIL import Image

# ---- CONFIG ----
INPUT_IMAGE = 'bank.png'  # Replace with your input
OUTPUT_TEXT = 'ocr_output.txt'
TESSERACT_LANG = 'nep'
TESSERACT_CONFIG = r'--psm 6 -l nep'

# ---- 1. Load Image as Grayscale ----
def load_image_grayscale(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE)

# ---- 2. Remove Watermark / Background ----
def remove_watermark(img):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    bg = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)
    return cv2.divide(img, bg, scale=255)

# ---- 3. Adaptive Thresholding ----
def adaptive_binarize(img):
    return cv2.adaptiveThreshold(
        img, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY, 15, 15)

# ---- 4. Remove Table Lines (Horizontal & Vertical) ----
def remove_lines(img):
    # Horizontal
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    h_lines = cv2.morphologyEx(img, cv2.MORPH_OPEN, h_kernel, iterations=1)
    no_h = cv2.subtract(img, h_lines)

    # Vertical
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    v_lines = cv2.morphologyEx(no_h, cv2.MORPH_OPEN, v_kernel, iterations=1)
    return cv2.subtract(no_h, v_lines)

# ---- 5. Skew Correction ----
def deskew(img):
    coords = np.column_stack(np.where(img > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = img.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

# ---- 6. Resize and Convert to 300 DPI PIL Image ----
def prepare_for_ocr(img_cv):
    # Resize to approx 300 DPI
    target_height = 3500  # ~A4 size at 300 DPI
    scale_factor = target_height / img_cv.shape[0]
    resized = cv2.resize(img_cv, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)

    # Convert to PIL Image
    pil_img = Image.fromarray(resized)
    pil_img.info['dpi'] = (300, 300)
    return pil_img

# ---- 7. OCR with Tesseract (Nepali) ----
def run_ocr(pil_image):
    return pytesseract.image_to_string(pil_image, config=TESSERACT_CONFIG)

# ---- 8. Pipeline ----
def process_image(path):
    img = load_image_grayscale(path)
    no_bg = remove_watermark(img)
    binary = adaptive_binarize(no_bg)
    clean = remove_lines(binary)
    aligned = deskew(clean)
    pil_ready = prepare_for_ocr(aligned)
    return run_ocr(pil_ready)

# ---- Run ----
if __name__ == "__main__":
    result = process_image(INPUT_IMAGE)
    print("\n===== OCR OUTPUT =====\n")
    print(result)
    with open(OUTPUT_TEXT, 'w', encoding='utf-8') as f:
        f.write(result)


TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [5]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os

# ==== CONFIG ====
# Optional: Path to tesseract executable (for Windows)
# Uncomment and set your actual path if needed
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

TESSERACT_LANG = 'nep'
TESSERACT_CONFIG = r'--psm 6 -l nep'

# ==== HELPER FUNCTIONS ====

def load_image_grayscale(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE)

def remove_watermark(img):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    bg = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)
    return cv2.divide(img, bg, scale=255)

def adaptive_binarize(img):
    return cv2.adaptiveThreshold(
        img, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY, 15, 15)

def remove_lines(img):
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    h_lines = cv2.morphologyEx(img, cv2.MORPH_OPEN, h_kernel, iterations=1)
    no_h = cv2.subtract(img, h_lines)

    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    v_lines = cv2.morphologyEx(no_h, cv2.MORPH_OPEN, v_kernel, iterations=1)
    return cv2.subtract(no_h, v_lines)

def deskew(img):
    coords = np.column_stack(np.where(img > 0))
    if coords.size == 0:
        return img  # Skip if no content
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = img.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def prepare_for_ocr(img_cv):
    target_height = 3500  # For 300 DPI output
    scale_factor = target_height / img_cv.shape[0]
    resized = cv2.resize(img_cv, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)
    pil_img = Image.fromarray(resized)
    pil_img.info['dpi'] = (300, 300)
    return pil_img

def extract_nepali_text(image_path, save_output=True, output_file='ocr_output.txt'):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    print(f"🔍 Processing: {image_path}")
    img = load_image_grayscale(image_path)
    no_bg = remove_watermark(img)
    binary = adaptive_binarize(no_bg)
    clean = remove_lines(binary)
    aligned = deskew(clean)
    pil_ready = prepare_for_ocr(aligned)

    print("🧠 Running OCR...")
    text = pytesseract.image_to_string(pil_ready, config=TESSERACT_CONFIG)

    if save_output:
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"📁 OCR result saved to: {output_file}")

    return text

# ==== MAIN ====

if __name__ == '__main__':
    image_path = 'bank.png'  # Replace with your image
    try:
        nepali_text = extract_nepali_text(image_path)
        print("\n✅ Extracted Nepali Text:\n")
        print(nepali_text)
    except Exception as e:
        print(f"❌ Error: {e}")


🔍 Processing: bank.png
🧠 Running OCR...
📁 OCR result saved to: ocr_output.txt

✅ Extracted Nepali Text:

| पत) चा १ हा . ... ॥ ॥ पि क कि छी बा
0 00 1 ॥ 0101 1010 (
॥1011110111111100111111000
॥ बह उ (३ क] ॥ बु म 0 छ ! 02 म छु छ मै। हज ॥1 0 ॥| ॥ | हित ॥ प्र त?
बध हठ $0 ई.. |) छाड ॥ 7 १
। | 1) | 10॥£ ||| 1011
140110104010110 02 | || 1010: ॥। ८,
बिते | क रछ) री ४ निति बि म त डजान दा



In [7]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os

# Optional: Only for Windows
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

TESSERACT_LANG = 'nep'
TESSERACT_CONFIG = r'--oem 3 --psm 4 -l nep'  # psm 4 = Assume column text, better for scanned docs

def preprocess_image(img_path):
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Median blur to remove noise, preserve edges
    blurred = cv2.medianBlur(gray, 3)

    # Histogram equalization for contrast
    equalized = cv2.equalizeHist(blurred)

    # Binarization using adaptive Gaussian
    thresh = cv2.adaptiveThreshold(
        equalized, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 21, 10
    )

    # Optional: Invert (Tesseract sometimes works better)
    inverted = cv2.bitwise_not(thresh)

    return inverted

def deskew_image(img):
    coords = np.column_stack(np.where(img > 0))
    if len(coords) == 0:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = img.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def upscale_for_ocr(img):
    # Upscale to improve recognition of small characters
    height = img.shape[0]
    scale = 3000 / height
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

def extract_nepali_text(image_path, save_output=True, output_file='ocr_output.txt'):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    print(f"🔍 Processing image: {image_path}")
    img = preprocess_image(image_path)
    deskewed = deskew_image(img)
    upscaled = upscale_for_ocr(deskewed)
    pil_img = Image.fromarray(upscaled)

    print("🧠 Running Tesseract OCR...")
    text = pytesseract.image_to_string(pil_img, config=TESSERACT_CONFIG)

    if save_output:
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"✅ OCR output saved to: {output_file}")

    return text

# Example usage
if __name__ == '__main__':
    image_path = 'bank.png'
    try:
        result = extract_nepali_text(image_path)
        print("\n📝 OCR Output:\n")
        print(result)
    except Exception as e:
        print(f"❌ Error: {e}")


🔍 Processing image: bank.png
🧠 Running Tesseract OCR...
✅ OCR output saved to: ocr_output.txt

📝 OCR Output:




In [8]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os

# If on Windows, uncomment:
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

TESSERACT_LANG = 'nep'
TESSERACT_CONFIG = '--oem 3 --psm 4 -l nep'

def preprocess_image(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # Clean background a little
    blurred = cv2.medianBlur(gray, 3)

    # Histogram equalization for contrast enhancement
    enhanced = cv2.equalizeHist(blurred)

    # Binarization using Otsu
    _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return thresh  # Do NOT invert

def deskew(img):
    coords = np.column_stack(np.where(img < 255))
    if len(coords) == 0:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = img.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def upscale(img, target_height=3000):
    scale = target_height / img.shape[0]
    return cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

def extract_nepali_text(image_path, debug=True):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    print(f"🔍 Processing: {image_path}")
    img = preprocess_image(image_path)
    deskewed = deskew(img)
    upscaled = upscale(deskewed)

    # Optional: Save debug image to visually confirm text quality
    if debug:
        cv2.imwrite('debug_output.png', upscaled)
        print("🖼️ Saved preprocessed image to 'debug_output.png'")

    pil_img = Image.fromarray(upscaled)
    print("🧠 Running Tesseract OCR...")

    text = pytesseract.image_to_string(pil_img, config=TESSERACT_CONFIG)

    if not text.strip():
        print("⚠️ OCR returned empty output. Try adjusting DPI or checking image contrast.")

    with open("ocr_output.txt", 'w', encoding='utf-8') as f:
        f.write(text)

    return text

# Run
if __name__ == '__main__':
    image_path = 'bank.png'
    output_text = extract_nepali_text(image_path)
    print("\n📄 OCR RESULT:\n")
    print(output_text)


🔍 Processing: bank.png
🖼️ Saved preprocessed image to 'debug_output.png'
🧠 Running Tesseract OCR...

📄 OCR RESULT:

१ २ , (. ७. ०० हि
|ह०4“आप्युति ५ ४. ककु9रा ईन, एन पि ०५ की हा बिह ।
“ ॥ हु ० " " हु हु क १" पिका [| १ व्य
9 ० ति , । ।
0 उता ता तशनमामकाक । ॥ .” १ . । द ,

श्योखर » कक



In [9]:
import cv2
import pytesseract
import os
from PIL import Image

# Path to tesseract.exe if on Windows
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def extract_nepali_text(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Step 1: Load and convert to grayscale
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Step 2: Slight denoising (if needed)
    gray = cv2.fastNlMeansDenoising(gray, h=10)

    # Step 3: Light binarization
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 4: Slight upscale (improve fine detail recognition)
    upscale = cv2.resize(binary, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)

    # Optional debug
    cv2.imwrite("cleaned.png", upscale)

    # Step 5: Convert to PIL and OCR
    pil_img = Image.fromarray(upscale)
    config = "--oem 3 --psm 4 -l nep"
    text = pytesseract.image_to_string(pil_img, config=config)

    with open("ocr_output.txt", 'w', encoding='utf-8') as f:
        f.write(text)

    return text

# Run
if __name__ == '__main__':
    image_path = 'bank.png'
    try:
        result = extract_nepali_text(image_path)
        print("\n✅ Final OCR Output:\n")
        print(result)
    except Exception as e:
        print(f"❌ Error: {e}")



✅ Final OCR Output:

१७४१ “01 ४
जाल राष्ट्र बैंक, “

तथा वित्तिय नियमन विभाग,
बालुवाटार, काठमाण्डौ |

मिति : २०८१/१० ४०४

आप्र
विषय :- -जैक खाता तथा लकर रोल्का गरि दिने सस्चन्धमा । २

आएको सन्दर्भमा पत्र प्रेषित गरिएको
ससस्याग्रस्त व्यवस्थापन तथा दायित्व भुक्तानीका लागि बैंक खाता तथा लकरहरु समेत सहकारी
ऐन, २०७४ को दफा ८४ र ११५ को उपदफा (१) को खण्ड (क) बमोजिम म
सदस्यहरुको खाता

- जिलाककन जिया]
(३ जलमा नका]
0 2 जा
6 र 7 | 20531513 11 1613 लोकमान बज्जचार्य

2

तुलसी नाययण शाक्य | मोति लाल शाक्य

१०४९/७३५७० | 2021312 | 13121 फुर्पा लामा न ब. लामा

हिटलरमान बच्जचार्य
ज्योति शाक्य

४६३७६/११९६७८
लक्ष्मी डोल्मा लामा
कमला थापा

१०१९५७ | 2978 | 18119

पा आण ___
ज्ञान ब, थापा
__
204711018 कृष्ण ब. खड्का देल ब. खड्का
शुसिला शिल्पकार १२ 11 २

2044110117 राम कृष्ण शिल्पकार |राम् बमन(ससुरा)
“२ | राम शरण सुवेदी ४०४८९ | 2021315 1 1315 प्रेम प्र.सुवेदी दिनानाथ सुबेदी

| पनलाल राजयला | राजथला तुईसिं राजथला



In [10]:
import cv2
import pytesseract
from PIL import Image
import numpy as np

def preprocess_and_ocr(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Invert if needed (text black, bg white)
    if np.mean(binary) < 127:
        binary = cv2.bitwise_not(binary)

    # Deskew
    coords = np.column_stack(np.where(binary > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = binary.shape
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    deskewed = cv2.warpAffine(binary, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    # Denoise lightly
    denoised = cv2.fastNlMeansDenoising(deskewed, h=10)

    # Upscale
    scaled = cv2.resize(denoised, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)

    # OCR
    pil_img = Image.fromarray(scaled)
    config = '--oem 3 --psm 4 -l nep'
    text = pytesseract.image_to_string(pil_img, config=config)
    return text

# Run
text = preprocess_and_ocr('bank.png')
print(text)


| | | |) | | "। 2?) ए|। २) ७० 00 ,.|,

। 3413 ३३ ॥

पि

1178
। ॥ । ॥ ॥
5३३३ ३
1३५३

111 |

बट
क्की 0200 0
| २1 ७| छै
ल्‌ क | 20 81 द्रट
७० प्र ति
छ
जो
॥। अ
11 । ॥ हि
| $। छ| छ | 8 ॥
४।। £1३।४। छु हौ $ ।
ण विय 5 5 ७ ७० 044 छ बु
डि हं खन ० टि प्रि ष्ण् गर
त ॥ । डु
सँ। 49 जै। ्ु दै
7

जअअ | बब

12८



In [11]:
import cv2
import pytesseract
import os
import numpy as np

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def mild_deskew(img):
    coords = np.column_stack(np.where(img > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 1.0:
        return img
    (h, w) = img.shape
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def extract_nepali_text(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Optional: mild denoise (comment out if no noise)
    gray = cv2.fastNlMeansDenoising(gray, h=10)

    # Mild deskew if needed
    gray = mild_deskew(gray)

    # OCR config
    custom_config = r'--oem 3 --psm 4 -l nep'

    text = pytesseract.image_to_string(gray, config=custom_config)
    return text

if __name__ == '__main__':
    image_path = 'bank.png'
    try:
        nepali_text = extract_nepali_text(image_path)
        print("✅ Extracted Nepali Text:\n")
        print(nepali_text)
    except Exception as e:
        print(f"❌ Error: {e}")


✅ Extracted Nepali Text:

क्र

111 $ ॥॥ | 1518 1
ं 4111 11 ॥ 2 1 |

| | | |) | | र)। ” ७ | ० ७ ।.|.

1110 17
000: || । || 1110
(43 05
॥॥| ॥॥ 1000
518 न 0 मु 1151 |
।॥। बु ॥॥ |1 2301 ई
0101 1010 मधे त ३ ३३ 94
५5014३ ॥। बँ। १३३ ती £27 ई
1111 1111 1111 ।

1110 ॥ ॥॥ । ३३३




In [12]:
import cv2
import pytesseract
import os

# OPTIONAL: For Windows users, specify the path to tesseract.exe
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def extract_nepali_text(image_path):
    # Check if image file exists
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Load the image
    image = cv2.imread(image_path)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Optional: Apply thresholding for better accuracy
    # gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

    # OCR: Extract Nepali text
    text = pytesseract.image_to_string(gray, lang='nep')

    return text

# Example usage
if __name__ == '__main__':
    image_path = 'bank.png'  # Replace with your image file
    try:
        nepali_text = extract_nepali_text(image_path)
        print("✅ Extracted Nepali Text:\n")
        print(nepali_text)
    except Exception as e:
        print(f"❌ Error: {e}")

✅ Extracted Nepali Text:

०१ 000
0600 “0 मिति : २०८१/१०/०४
श्री नेपाल राष्ट्र बैंक,

बैंक तथा वित्तिय नियमन विभाग,
बालुवाटार, काठमाण्डौ ।
आप्‌
विषय :- -जैंक खाता तथा लकर रोक्का गरि दिने सम्बन्धमा । तथा लकर रोक्का गरि दिने । तीर

उपर्युक्त सम्बन्धमा नेपाल सरकार भूमि व्यवस्था, सहकारी तथा गरिबी निवारण मन्त्रालयको माननीय मन्त्रीस्तरको
मिति २०८०।११।०४ को निर्णयबाट पशुपति सेभिङ एण्ड क्रेडिट को-अपरेटिभ लिमिटेडलाई समस्याग्रस्त घोषणा गरी उक्त
संस्थाको सम्पत्ति व्यवस्थापन तथा दायित्व भुक्तानीको लागि यस समिति समक्ष लेखि आएको सन्दर्भमा पत्र प्रेषित गरिएको
छ । समस्याग्रस्त संस्थाको सम्पत्ति व्यवस्थापन तथा दायित्व भुक्तानीका लागि बैंक खाता तथा लकरहरु समेत सहकारी

सिने [करणीको नाम ७उउँकिनक जारी मिति 'पनक्याक्क नाम कन सिक नाम
1 | नवराज घिमिरे बिक) ७ रन ७४५५ 20611819 मानब, विश्वकर्मा अखर ब. विश्वकर्मा
| ३ । ति ता बैद्य छि लयाल। डे 20441514 0 छा बैद्य भुवन बैद्य
| जनै ताम | तामाङ्ग हित हा २१०२६/४ 20641116 निमा सिं तामाङ्ग धम्बल सिं तामाङ्ग
नि मदन कुमार श्रेष्ठ ८९७४० 120612190 1 112110 लोकब श्रेष्ठ क्षेत

In [14]:
import cv2
import pytesseract
import os

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def enhance_contrast(gray):
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    return clahe.apply(gray)

def extract_nepali_text(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = enhance_contrast(gray)  # Improve contrast

    custom_config = r'--oem 3 --psm 6 -l nep'
    text = pytesseract.image_to_string(gray, config=custom_config)

    return text

if __name__ == '__main__':
    image_path = 'bank.png'
    try:
        nepali_text = extract_nepali_text(image_path)
        print("✅ Extracted Nepali Text:\n")
        print(nepali_text)
    except Exception as e:
        print(f"❌ Error: {e}")


✅ Extracted Nepali Text:

टि हि” 0
खत क टि मिति : २०८१/१०/०४
थरी नेपाल राष्ट्र बैंक,
बैंक तथा वित्तिय नियमन विभाग,
वालुवाटार, काठमाण्डौ । जा प्‌
विषय :- 6000
उपर्युक्त सम्बन्धमा नेपाल सरकार भूमि व्यवस्था, सहकारी तथा गरिबी निवारण मन्त्रालयको माननीय मन्त्रीस्तरको
मिति २०८०।११।०४ को निर्णयबाट पशुपति सेभिङ एण्ड क्रेडिट को-अपरेटिभ लिमिटेडलाई समस्याग्रस्त घोषणा गरी उक्त
संस्थाको सम्पत्ति व्यवस्थापन तथा दायित्व भुक्तानीको लागि यस समिति समक्ष लेखि आएको सन्दर्भमा पत्र प्रेषित गरिएको
छ । समस्याग्रस्त संस्थाको सम्पत्ति व्यवस्थापन तथा दायित्व भुक्तानीका लागि बैंक खाता तथा लकरहरु समेत सहकारी
ऐन, २०७४ को दफा ८४ २ ११५ को उपदफा (१) को खण्ड (क) बमोजिम तपसीलमा उल्लेखित संस्थाका क्रणी
सदस्यहरुको खाता तथा लकर रोक्का राखी सो को जानकारी यस समितिलाई दिनु हुन निर्णयानुसार अनुरोध गरिन्छ ।
जपसील
३ चिक कमा जिया]
जन नमा
1011 आहया जिया
७ 1001 आना
५ नका]
विक
8 १०५४९/७३५७० याय फुर्पा लामा पन्च ब. लामा
कै
क -किककाहतका
कि जिन
० ला गन त या रि



In [15]:
import cv2
import pytesseract
import easyocr

def preprocess_variants(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Variant 1: raw grayscale
    yield gray

    # Variant 2: CLAHE contrast enhanced
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    yield clahe.apply(gray)

    # Variant 3: adaptive threshold
    yield cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 15)

    # Variant 4: Otsu threshold
    _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    yield otsu

def run_tesseract(image, psm=6):
    config = f'--oem 3 --psm {psm} -l nep'
    return pytesseract.image_to_string(image, config=config)

def run_easyocr(image_path):
    reader = easyocr.Reader(['ne'], gpu=False)
    results = reader.readtext(image_path, detail=0)
    return '\n'.join(results)

def combine_texts(texts):
    # Simple heuristic: return the longest text (you can improve this)
    return max(texts, key=len)

if __name__ == '__main__':
    image_path = 'bank.png'
    ocr_texts = []

    for variant_img in preprocess_variants(image_path):
        text = run_tesseract(variant_img)
        ocr_texts.append(text)

    # Add easyOCR result as well
    easy_text = run_easyocr(image_path)
    ocr_texts.append(easy_text)

    combined_result = combine_texts(ocr_texts)
    print("📝 Combined OCR Result:\n", combined_result)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\praka\AppData\Local\anaconda3\envs\python\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\praka\AppData\Local\anaconda3\envs\python\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\praka\AppData\Local\anaconda3\envs\python\Lib\site-packages\ipykernel\kernelapp.py", line 739, in

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

RuntimeError: Numpy is not available

In [16]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [17]:
data = pd.read_csv("CUSTOMER_SAMPLE.csv")

In [23]:
data.columns

Index(['CUS_ID', 'CUS_NAME', 'CUS_LEG_DOC', 'CUS_LEG_ID', 'CUS_LEG_ISS_DT',
       'CUS_FATHER', 'CUS_GRANDFATHER', 'CUS_ADD', 'CUS_DISTRICT'],
      dtype='object')

In [26]:
data['CUS_ID'].count()

np.int64(20000)

In [27]:
data

,CUS_ID,CUS_NAME,CUS_LEG_DOC,CUS_LEG_ID,CUS_LEG_ISS_DT,CUS_FATHER,CUS_GRANDFATHER,CUS_ADD,CUS_DISTRICT
0,784120,NARAYAN BAHADUR MAGAR,CITIZENSHIP.NO,86029,20540823,BHAKTA BDR,LAL BDR,DHUMTHANG 7,SINDHUPALCHOWK
1,5209647,SONMATI MALLAH,CITIZENSHIP.NO,37017401826,20740212,BUCHUN MALLAH,BUDDHU MALLAH,MAYADEVI 03 HARNAM PUR,RUPANDEHI 05
2,1282862,SABITRA SUNAR,CITIZENSHIP.NO,40-01-73-04962,20730928,ECHHE KAMI,KULE KAMI,SA.NA.PA. 02,ARGHAKHANCHI
3,275027,SHANTI ACHARYA,CITIZENSHIP.NO,11724,20450432,UTTAM MANI ACHARYA,CHAKRAPANI ACHARYA,HETAUDA -04,NaN
4,1497712,SAJMUL KHATUN,CITIZENSHIP.NO,4075/501,20460212,JWAR HAJI,NIJHAR MIYA,RATUWAMAI 02,MORANG
...,...,...,...,...,...,...,...,...,...
19995,3064587,MINA SHRESTHA MALLA,CITIZENSHIP.NO,55126/10850,20661229,KRISHNA BAHADUR SHRESTHA,PREM PRASAD SHRESTHA,KAPURKOT RURAL MUNICIPALITY,SALYAN
19996,1446220,TARA BAHADUR NEUPANE,CITIZENSHIP.NO,1300/044,20570823,KEDAR NATH NEUPANE,LATE DATTU NEUPANE,PITHUWA-02 CHITWAN,CHITWAN
19997,5483079,TEJ BAHADUR BHANDARI,CITIZENSHIP.NO,12253/193,20430322,NaN,NaN,RATUWAMAI 10,NaN
19998,1688405,NANDAKALA SARKI,CITIZENSHIP.NO,613030/54/41,20631205,PAUMALE SARKI,DHARMA SARKI,NARAHARINATH09,KALIKOT


In [29]:
data.describe()

,CUS_ID
count,2.000000e+04
mean,2.014263e+06
std,1.845740e+06
min,1.700000e+01
25%,6.369690e+05
50%,1.341372e+06
75%,3.027878e+06
max,9.480510e+06


In [31]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   CUS_ID           20000 non-null  int64 
 1   CUS_NAME         20000 non-null  object
 2   CUS_LEG_DOC      16895 non-null  object
 3   CUS_LEG_ID       16834 non-null  object
 4   CUS_LEG_ISS_DT   16669 non-null  object
 5   CUS_FATHER       13761 non-null  object
 6   CUS_GRANDFATHER  11295 non-null  object
 7   CUS_ADD          19848 non-null  object
 8   CUS_DISTRICT     15874 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.4+ MB
